# Etapa 3 — Feature Engineering

**Objetivo:** Construir o conjunto de features para o modelo XGBoost, calculado sobre a **série completa** antes do split temporal.

**Metodologia de leakage:** Lags e janelas móveis usam exclusivamente lookback de valores passados em relação a cada ponto t. Não há acesso a valores futuros. O split treino/teste é aplicado após a construção das features (ver DECISOES.md — entrada sobre ordem feature engineering/split).

**Entrada:** `data/processed/beverages_daily.csv`  
**Saída:** `data/processed/beverages_features.csv`

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

PROCESSED_DIR = '../data/processed/'

daily = pd.read_csv(PROCESSED_DIR + 'beverages_daily.csv', parse_dates=['date'])
daily = daily.sort_values('date').reset_index(drop=True)
print(f'Série carregada: {len(daily)} observações')
print(f'Colunas: {list(daily.columns)}')

Série carregada: 1684 observações
Colunas: ['date', 'sales', 'onpromotion', 'oil_price', 'is_national_holiday']


## 3.1 Features de calendário

In [2]:
df = daily.copy()

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.dayofweek          # 0=Seg, 6=Dom
df['day_of_year'] = df['date'].dt.dayofyear
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['date'].dt.quarter

print('Features de calendário criadas:')
print(['year','month','day','day_of_week','day_of_year','week_of_year','quarter'])
df[['date','year','month','day','day_of_week','day_of_year','week_of_year','quarter']].head()

Features de calendário criadas:
['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter']


,date,year,month,day,day_of_week,day_of_year,week_of_year,quarter
0,2013-01-01,2013,1,1,1,1,1,1
1,2013-01-02,2013,1,2,2,2,1,1
2,2013-01-03,2013,1,3,3,3,1,1
3,2013-01-04,2013,1,4,4,4,1,1
4,2013-01-05,2013,1,5,5,5,1,1


## 3.2 Features de lag

In [3]:
# Lags da variável alvo (sales)
# Calculados sobre a série completa — sem leakage pois usam apenas valores passados
for lag in [7, 14, 30, 365]:
    df[f'lag_{lag}'] = df['sales'].shift(lag)

print('Lags criados: lag_7, lag_14, lag_30, lag_365')
print(f'NaN gerados por lag_365: {df["lag_365"].isna().sum()} (primeiros 365 dias)')

Lags criados: lag_7, lag_14, lag_30, lag_365
NaN gerados por lag_365: 365 (primeiros 365 dias)


## 3.3 Features de janelas móveis (rolling)

In [4]:
# Rolling sobre série de sales
# min_periods=1 evita NaN extras; shift(1) garante que a janela não inclui o valor atual (sem leakage)
sales_shifted = df['sales'].shift(1)  # usar somente valores até t-1

df['rolling_7_mean'] = sales_shifted.rolling(window=7, min_periods=7).mean()
df['rolling_30_mean'] = sales_shifted.rolling(window=30, min_periods=30).mean()
df['rolling_7_std'] = sales_shifted.rolling(window=7, min_periods=7).std()
df['rolling_30_std'] = sales_shifted.rolling(window=30, min_periods=30).std()

print('Rolling features criadas: rolling_7_mean, rolling_30_mean, rolling_7_std, rolling_30_std')
print(f'NaN em rolling_30_mean: {df["rolling_30_mean"].isna().sum()}')

Rolling features criadas: rolling_7_mean, rolling_30_mean, rolling_7_std, rolling_30_std
NaN em rolling_30_mean: 30


## 3.4 Verificação de leakage

In [5]:
# Verificação de leakage:
# rolling_7_mean[i] = mean(sales_shifted[i-6:i+1]) = mean(sales[i-7:i])
# Em .loc (label-based, inclusive): df.loc[i-7:i-1, 'sales'].mean()

idx = 400  # dia 400 da série
print(f'Verificação de leakage para o índice {idx} (data: {df.loc[idx, "date"].date()}):')
print(f'  sales[{idx}] = {df.loc[idx, "sales"]:.0f}')
print(f'  lag_7[{idx}] deve ser sales[{idx-7}] = {df.loc[idx-7, "sales"]:.0f}')
print(f'  lag_7[{idx}] calculado = {df.loc[idx, "lag_7"]:.0f}')
assert df.loc[idx, 'lag_7'] == df.loc[idx-7, 'sales'], 'ERRO: lag_7 com leakage!'
print(f'  rolling_7_mean[{idx}] = {df.loc[idx, "rolling_7_mean"]:.1f}')
# Correto: mean(sales[idx-7], ..., sales[idx-1]) — 7 valores, nenhum é sales[idx]
manual_rolling = df.loc[idx-7:idx-1, 'sales'].mean()
print(f'  Manual mean(sales[{idx-7}:{idx-1}]) = {manual_rolling:.1f}')
assert abs(df.loc[idx, 'rolling_7_mean'] - manual_rolling) < 0.01, 'ERRO: rolling com leakage!'
print('  ✓ Sem leakage confirmado — rolling não inclui sales[t], apenas sales[t-7] a sales[t-1]')

Verificação de leakage para o índice 400 (data: 2014-02-06):
  sales[400] = 55583
  lag_7[400] deve ser sales[393] = 95195
  lag_7[400] calculado = 95195
  rolling_7_mean[400] = 84358.6
  Manual mean(sales[393:399]) = 84358.6
  ✓ Sem leakage confirmado — rolling não inclui sales[t], apenas sales[t-7] a sales[t-1]


## 3.5 Remover NaN e salvar

In [6]:
print(f'Shape antes de remover NaN: {df.shape}')
print(f'NaN por coluna (apenas colunas com NaN):')
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 0])

df_clean = df.dropna().reset_index(drop=True)
print(f'\nShape após remover NaN: {df_clean.shape}')
print(f'Linhas removidas: {len(df) - len(df_clean)} (primeiros ~365 dias por lag_365)')
print(f'Período restante: {df_clean["date"].min().date()} a {df_clean["date"].max().date()}')

Shape antes de remover NaN: (1684, 20)
NaN por coluna (apenas colunas com NaN):
lag_7                7
lag_14              14
lag_30              30
lag_365            365
rolling_7_mean       7
rolling_30_mean     30
rolling_7_std        7
rolling_30_std      30
dtype: int64

Shape após remover NaN: (1319, 20)
Linhas removidas: 365 (primeiros ~365 dias por lag_365)
Período restante: 2014-01-02 a 2017-08-15


In [7]:
feature_cols = [
    'date', 'sales',
    # Calendário
    'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter',
    # Lags
    'lag_7', 'lag_14', 'lag_30', 'lag_365',
    # Rolling
    'rolling_7_mean', 'rolling_30_mean', 'rolling_7_std', 'rolling_30_std',
    # Exógenas
    'onpromotion', 'oil_price', 'is_national_holiday'
]

df_final = df_clean[feature_cols].copy()
print(f'Features finais: {len(feature_cols) - 2} features + date + sales')
print(f'Colunas: {feature_cols}')
df_final.head()

Features finais: 18 features + date + sales
Colunas: ['date', 'sales', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'quarter', 'lag_7', 'lag_14', 'lag_30', 'lag_365', 'rolling_7_mean', 'rolling_30_mean', 'rolling_7_std', 'rolling_30_std', 'onpromotion', 'oil_price', 'is_national_holiday']


,date,sales,year,month,day,day_of_week,day_of_year,week_of_year,quarter,lag_7,lag_14,lag_30,lag_365,rolling_7_mean,rolling_30_mean,rolling_7_std,rolling_30_std,onpromotion,oil_price,is_national_holiday
0,2014-01-02,172933.0,2014,1,2,3,2,1,1,73168.0,63989.0,69163.0,810.0,72554.428571,71272.366667,32152.036995,19374.649424,0,95.14,0
1,2014-01-03,146684.0,2014,1,3,4,3,1,1,77409.0,62692.0,57259.0,72092.0,86806.571429,74731.366667,49759.650651,26818.308646,0,93.66,0
2,2014-01-04,213643.0,2014,1,4,5,4,1,1,88614.0,70487.0,55770.0,52105.0,96703.000000,77712.200000,54264.098033,29631.510201,0,93.66,0
3,2014-01-05,222012.0,2014,1,5,6,5,1,1,78396.0,91862.0,54490.0,54167.0,114564.285714,82974.633333,69574.788297,38339.540954,0,93.66,0
4,2014-01-06,134433.0,2014,1,6,0,6,2,1,96102.0,89825.0,67538.0,77818.0,135080.857143,88558.700000,77818.448424,45566.265726,0,93.12,0


In [8]:
# Estatísticas das features
print('Correlação das features com sales:')
numeric_cols = [c for c in feature_cols if c not in ['date', 'sales']]
correlations = df_final[numeric_cols].corrwith(df_final['sales']).sort_values(ascending=False)
print(correlations.round(3))

Correlação das features com sales:
lag_7                  0.820
lag_14                 0.790
rolling_7_mean         0.715
rolling_30_mean        0.661
rolling_7_std          0.542
onpromotion            0.513
year                   0.511
rolling_30_std         0.488
day_of_week            0.369
lag_365                0.308
lag_30                 0.305
month                  0.209
quarter                0.204
day_of_year            0.202
week_of_year           0.180
is_national_holiday    0.099
day                   -0.077
oil_price             -0.433
dtype: float64


In [9]:
df_final.to_csv(PROCESSED_DIR + 'beverages_features.csv', index=False)
print(f'Arquivo salvo: {PROCESSED_DIR}beverages_features.csv')
print(f'Shape final: {df_final.shape}')
print(f'Sem NaN: {df_final.isna().sum().sum() == 0}')

Arquivo salvo: ../data/processed/beverages_features.csv
Shape final: (1319, 20)
Sem NaN: True
